# 02.3 — Alpine resort spatial features, hotel infrastructure and EDA

This notebook starts from the Alpine-only 280-resort sample created in 02.2. It refreshes coordinates rather than trusting the old geocoding cache, using explicit source coordinates when available and otherwise Nominatim, Photon and Open-Meteo geocoding. The selected coordinate is labelled as a resort point or nearest access-village point so its precision is clear.

It then calculates straight-line distances to six major cities, counts mapped 4- and 5-star hotels within 10 km using small Overpass batches, and produces descriptive statistics and plots. External-service failures are recorded per batch; they never trigger fake coordinates, zero counts or proxy values.

In [ ]:
import os
import re
import time
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

DATA_DIR = "../data/processed"
INPUT_CSV = os.path.join(DATA_DIR, "resorts_clean.csv")
FEATURES_CSV = os.path.join(DATA_DIR, "resorts_features.csv")
GEOCODE_CACHE_CSV = os.path.join(DATA_DIR, "resorts_geocoded_cache.csv")
HOTEL_CACHE_CSV = os.path.join(DATA_DIR, "hotels_cache.csv")
COVERAGE_CSV = os.path.join(DATA_DIR, "resorts_feature_coverage.csv")
os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)
EXPECTED_COUNTRIES = {"CH", "IT", "AT", "FR"}
if len(df) != 280 or set(df["country"].unique()) != EXPECTED_COUNTRIES:
    raise ValueError("Run 02.2 first: expected 280 Alpine resorts in CH, IT, AT and FR.")
if not df.groupby("country").size().eq(70).all():
    raise ValueError("Expected 70 Alpine resorts per country.")
if not df["alpine_scope_verified"].fillna(False).all():
    raise ValueError("The input contains records without an Alpine source-scope flag.")
if not df["url"].is_unique:
    raise ValueError("Duplicate resort URLs are not allowed.")

HEADERS = {
    "User-Agent": "ZHAW-AlpineSkiResearch/1.0 (academic project; repository contact)",
    "Accept-Language": "en",
}
COUNTRY_NAMES = {
    "CH": "Switzerland", "IT": "Italy", "AT": "Austria", "FR": "France"
}
COUNTRY_CODES = {"CH": "ch", "IT": "it", "AT": "at", "FR": "fr"}
# Broad country envelopes used only to reject obviously misplaced geocoder results.
# Alpine membership itself comes from the source Alpine directories, not these boxes.
COUNTRY_BOUNDS = {
    "CH": (45.7, 47.9, 5.8, 10.7),
    "IT": (44.0, 47.3, 6.0, 13.0),
    "AT": (46.3, 49.1, 9.4, 17.2),
    "FR": (43.5, 46.7, 5.0, 7.9),
}
NOMINATIM_VIEWBOX = {
    "CH": "5.8,47.9,10.7,45.7",
    "IT": "6.0,47.3,13.0,44.0",
    "AT": "9.4,49.1,17.2,46.3",
    "FR": "5.0,46.7,7.9,43.5",
}
CITIES = {
    "zurich": (47.3769, 8.5417),
    "geneva": (46.2044, 6.1432),
    "milan": (45.4642, 9.1900),
    "turin": (45.0703, 7.6869),
    "munich": (48.1351, 11.5820),
    "lyon": (45.7640, 4.8357),
}

session = requests.Session()
session.headers.update(HEADERS)


def normalise_text(value):
    value = str(value or "").replace("\u200b", "").replace("\u200c", "").replace("\u200d", "")
    value = value.replace("\u2060", "").replace("\ufeff", "").replace("’", "'")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    return re.sub(r"\s+", " ", value).strip()


def inside_country_bounds(lat, lon, country):
    south, north, west, east = COUNTRY_BOUNDS[country]
    return south <= lat <= north and west <= lon <= east


def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))


def geocode_nominatim(query, country):
    response = session.get(
        "https://nominatim.openstreetmap.org/search",
        params={
            "q": query, "format": "jsonv2", "limit": 5,
            "addressdetails": 1, "countrycodes": COUNTRY_CODES[country],
            "viewbox": NOMINATIM_VIEWBOX[country], "bounded": 1,
        },
        timeout=25,
    )
    response.raise_for_status()
    for item in response.json():
        address = item.get("address", {})
        code = str(address.get("country_code", "")).lower()
        lat, lon = float(item["lat"]), float(item["lon"])
        if code == COUNTRY_CODES[country] and inside_country_bounds(lat, lon, country):
            return {
                "latitude": lat, "longitude": lon,
                "provider": "Nominatim", "display_name": item.get("display_name"),
                "osm_type": item.get("osm_type"), "osm_id": item.get("osm_id"),
                "geocode_class": item.get("class"), "geocode_type": item.get("type"),
            }
    return None


def geocode_photon(query, country):
    response = session.get(
        "https://photon.komoot.io/api/",
        params={"q": query, "limit": 5, "lang": "en"},
        timeout=25,
    )
    response.raise_for_status()
    for feature in response.json().get("features", []):
        props = feature.get("properties", {})
        country_name = normalise_text(props.get("country", "")).lower()
        if country_name != COUNTRY_NAMES[country].lower():
            continue
        lon, lat = feature.get("geometry", {}).get("coordinates", [np.nan, np.nan])
        lat, lon = float(lat), float(lon)
        if inside_country_bounds(lat, lon, country):
            return {
                "latitude": lat, "longitude": lon,
                "provider": "Photon", "display_name": props.get("name"),
                "osm_type": props.get("osm_value"), "osm_id": props.get("osm_id"),
                "geocode_class": props.get("osm_key"), "geocode_type": props.get("type"),
            }
    return None


def geocode_open_meteo(query, country):
    response = session.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": query, "count": 10, "language": "en", "format": "json"},
        timeout=25,
    )
    response.raise_for_status()
    for item in response.json().get("results", []):
        code = str(item.get("country_code", "")).lower()
        lat, lon = float(item["latitude"]), float(item["longitude"])
        if code == COUNTRY_CODES[country] and inside_country_bounds(lat, lon, country):
            return {
                "latitude": lat, "longitude": lon,
                "provider": "Open-Meteo Geocoding",
                "display_name": ", ".join(str(item.get(k)) for k in ["name", "admin1", "country"] if item.get(k)),
                "osm_type": None, "osm_id": None,
                "geocode_class": item.get("feature_code"), "geocode_type": item.get("feature_code"),
            }
    return None


def resolve_resort_coordinate(row):
    country = row["country"]
    source_lat, source_lon = row.get("source_latitude"), row.get("source_longitude")
    if pd.notna(source_lat) and pd.notna(source_lon):
        source_lat, source_lon = float(source_lat), float(source_lon)
        if inside_country_bounds(source_lat, source_lon, country):
            return {
                "latitude": source_lat, "longitude": source_lon,
                "provider": "Skiresort detail metadata",
                "display_name": row["resort_name"], "osm_type": None, "osm_id": None,
                "geocode_class": None, "geocode_type": None,
                "query": row["url"], "coordinate_reference": "resort page coordinate",
                "status": "matched",
            }

    village = row.get("nearest_access_village")
    candidates = []
    if pd.notna(village) and str(village).strip():
        candidates.append((f"{row['resort_name']}, {village}, {COUNTRY_NAMES[country]}", "resort + nearest access village"))
        candidates.append((f"{village}, {COUNTRY_NAMES[country]}", "nearest access village"))
    candidates.append((f"{row['resort_name']}, {COUNTRY_NAMES[country]}, Alps", "resort name"))

    for query, reference in candidates:
        try:
            result = geocode_nominatim(query, country)
            time.sleep(1.1)  # Respect the public Nominatim usage policy.
            if result:
                return {**result, "query": query, "coordinate_reference": reference, "status": "matched"}
        except requests.RequestException as exc:
            last_error = type(exc).__name__
            time.sleep(1.0)

    for query, reference in candidates:
        try:
            result = geocode_photon(query, country)
            if result:
                return {**result, "query": query, "coordinate_reference": reference, "status": "matched"}
        except requests.RequestException:
            pass

    for query, reference in candidates:
        try:
            result = geocode_open_meteo(query, country)
            if result:
                return {**result, "query": query, "coordinate_reference": reference, "status": "matched"}
        except requests.RequestException:
            pass

    return {
        "latitude": np.nan, "longitude": np.nan, "provider": None,
        "display_name": None, "osm_type": None, "osm_id": None,
        "geocode_class": None, "geocode_type": None,
        "query": candidates[0][0] if candidates else row["resort_name"],
        "coordinate_reference": None,
        "status": "not_found",
    }


## 1. Refresh resort coordinates from multiple geocoding sources

The previous cache is not reused because some legacy coordinates were not independently validated. We first use coordinates explicitly embedded in the Skiresort detail page. Otherwise, we search the resort and its nearest access village using Nominatim, then Photon and Open-Meteo as fallbacks. Results must match the expected country and a broad country envelope; unresolved locations remain empty.

In [ ]:
geocode_records = []
for index, row in df.iterrows():
    result = resolve_resort_coordinate(row)
    geocode_records.append({
        "resort_name": row["resort_name"],
        "country": row["country"],
        **result,
        "geocode_retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    })
    if (index + 1) % 25 == 0:
        print(f"Geocoded {index + 1}/{len(df)} resorts")

geocodes = pd.DataFrame(geocode_records)
df = df.drop(columns=["latitude", "longitude", "geocode_source"], errors="ignore")
df = df.merge(
    geocodes[[
        "resort_name", "country", "latitude", "longitude", "provider",
        "display_name", "osm_type", "osm_id", "geocode_class", "geocode_type",
        "query", "coordinate_reference", "status", "geocode_retrieved_at_utc",
    ]],
    on=["resort_name", "country"], how="left", validate="one_to_one",
)
df = df.rename(columns={
    "provider": "geocode_provider",
    "display_name": "geocode_display_name",
    "query": "geocode_query",
    "status": "geocode_status",
})
df["geocode_source"] = df["geocode_provider"]
geocodes.to_csv(GEOCODE_CACHE_CSV, index=False)

print("Coordinates available:", int(df[["latitude", "longitude"]].notna().all(axis=1).sum()), "/", len(df))
print("Coordinate provider:")
print(df["geocode_provider"].value_counts(dropna=False))
display(df.loc[df["latitude"].isna(), ["resort_name", "country", "nearest_access_village"]])


## 2. Calculate distances to major cities

For each resort we calculate straight-line great-circle distances to Zurich, Geneva, Milan, Turin, Munich and Lyon. These are geographic distances, not road or public-transport travel times. Distances are calculated only when a validated resort/access-village coordinate is available.

In [ ]:
for city, (city_lat, city_lon) in CITIES.items():
    df[f"dist_{city}_km"] = [
        round(float(haversine_km(lat, lon, city_lat, city_lon)), 1)
        if pd.notna(lat) and pd.notna(lon) else np.nan
        for lat, lon in zip(df["latitude"], df["longitude"])
    ]

distance_columns = [f"dist_{city}_km" for city in CITIES]
df["dist_nearest_city_km"] = df[distance_columns].min(axis=1)
print("Resorts with nearest-city distance:", int(df["dist_nearest_city_km"].notna().sum()), "/", len(df))
display(df[["resort_name", "country", "coordinate_reference", "dist_nearest_city_km"]].head(10))


## 3. Count mapped 4- and 5-star hotels within 10 km

A single large Alpine bounding-box query is unreliable and can overload public Overpass servers. Instead, resorts are processed in small batches. Each query requests only mapped hotel objects within 10 km of the batch's resort points. We retry across several public Overpass instances; a failed batch is marked unavailable and does not stop the rest of the notebook.

Counts are split into 4-star and 5-star objects. They describe OpenStreetMap objects with explicit 4/5-star tags, not an official census of unique hotel businesses. Missing results are not converted to zero.

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://lz4.overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
HOTEL_RADIUS_METRES = 10000
HOTEL_BATCH_SIZE = 10
hotel_columns = [
    "osm_type", "osm_id", "hotel_name", "stars", "latitude", "longitude",
    "query_batch_id", "retrieved_at_utc",
]
hotel_rows = []
df["hotels_4_star_10km"] = np.nan
df["hotels_5_star_10km"] = np.nan
df["hotels_4_5_star_10km"] = np.nan
df["hotel_query_status"] = np.where(
    df[["latitude", "longitude"]].notna().all(axis=1),
    "pending", "no_coordinates"
)
df["hotel_query_batch"] = pd.NA
df["hotel_data_source"] = pd.NA

valid_indices = df.index[df[["latitude", "longitude"]].notna().all(axis=1)].tolist()
hotel_batch_results = []

for batch_number, start in enumerate(range(0, len(valid_indices), HOTEL_BATCH_SIZE), start=1):
    batch_indices = valid_indices[start:start + HOTEL_BATCH_SIZE]
    selectors = []
    for idx in batch_indices:
        resort = df.loc[idx]
        selectors.append(
            f'nwr(around:{HOTEL_RADIUS_METRES},{float(resort["latitude"]):.6f},'
            f'{float(resort["longitude"]):.6f})["tourism"="hotel"]["stars"~"^(4|5)(\\.0)?$"];'
        )
    query = "[out:json][timeout:120];\n(\n" + "\n".join(selectors) + "\n);\nout center tags;"

    elements = None
    last_error = None
    endpoint_order = OVERPASS_ENDPOINTS[batch_number % len(OVERPASS_ENDPOINTS):] + OVERPASS_ENDPOINTS[:batch_number % len(OVERPASS_ENDPOINTS)]
    for endpoint in endpoint_order:
        for attempt in range(2):
            try:
                response = session.post(
                    endpoint, data={"data": query}, timeout=150
                )
                response.raise_for_status()
                payload = response.json()
                elements = payload.get("elements", [])
                break
            except (requests.RequestException, ValueError) as exc:
                last_error = f"{endpoint}: {type(exc).__name__}"
                time.sleep(2 + attempt * 3)
        if elements is not None:
            break

    df.loc[batch_indices, "hotel_query_batch"] = batch_number
    if elements is None:
        df.loc[batch_indices, "hotel_query_status"] = "unavailable"
        hotel_batch_results.append({
            "batch": batch_number, "resorts": len(batch_indices),
            "status": "unavailable", "hotel_objects": np.nan,
            "error": last_error,
        })
        print(f"[WARN] Overpass batch {batch_number}: unavailable ({last_error})")
        continue

    batch_hotels = []
    for element in elements:
        tags = element.get("tags", {})
        stars = str(tags.get("stars", "")).strip()
        if not re.fullmatch(r"[45](?:\\.0)?", stars):
            continue
        lat = element.get("lat", element.get("center", {}).get("lat"))
        lon = element.get("lon", element.get("center", {}).get("lon"))
        if lat is None or lon is None:
            continue
        batch_hotels.append({
            "osm_type": element.get("type"),
            "osm_id": element.get("id"),
            "hotel_name": tags.get("name"),
            "stars": stars,
            "latitude": float(lat),
            "longitude": float(lon),
            "query_batch_id": batch_number,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
        })
    batch_hotels_df = pd.DataFrame(batch_hotels, columns=hotel_columns)
    if not batch_hotels_df.empty:
        batch_hotels_df = batch_hotels_df.drop_duplicates(subset=["osm_type", "osm_id"])
        hotel_rows.extend(batch_hotels_df.to_dict("records"))

    for idx in batch_indices:
        resort = df.loc[idx]
        if batch_hotels_df.empty:
            count4 = count5 = 0
        else:
            distances = haversine_km(
                float(resort["latitude"]), float(resort["longitude"]),
                batch_hotels_df["latitude"].to_numpy(),
                batch_hotels_df["longitude"].to_numpy(),
            )
            stars_numeric = pd.to_numeric(batch_hotels_df["stars"], errors="coerce").to_numpy()
            count4 = int(np.sum((distances <= 10.0) & (stars_numeric == 4)))
            count5 = int(np.sum((distances <= 10.0) & (stars_numeric == 5)))
        df.loc[idx, "hotels_4_star_10km"] = count4
        df.loc[idx, "hotels_5_star_10km"] = count5
        df.loc[idx, "hotels_4_5_star_10km"] = count4 + count5
        df.loc[idx, "hotel_query_status"] = "success"
        df.loc[idx, "hotel_data_source"] = "OpenStreetMap Overpass"

    hotel_batch_results.append({
        "batch": batch_number, "resorts": len(batch_indices),
        "status": "success", "hotel_objects": len(batch_hotels_df),
        "error": None,
    })
    print(f"Overpass batch {batch_number}: {len(batch_indices)} resorts, {len(batch_hotels_df)} hotel objects")
    time.sleep(1.0)

hotels = pd.DataFrame(hotel_rows, columns=hotel_columns)
if not hotels.empty:
    hotels = hotels.drop_duplicates(subset=["osm_type", "osm_id"]).reset_index(drop=True)
hotels.to_csv(HOTEL_CACHE_CSV, index=False)
batch_summary = pd.DataFrame(hotel_batch_results)
batch_summary.to_csv(os.path.join(DATA_DIR, "overpass_batch_log.csv"), index=False)

print("Hotel counts available:", int(df["hotels_4_5_star_10km"].notna().sum()), "/", len(df))
print("Overpass hotel objects:", len(hotels))
print("Successful batches:", int((batch_summary["status"] == "success").sum()), "/", len(batch_summary))


## 4. EDA — prices, resort size, snowmaking and spatial distribution

The EDA uses only observed source values. It compares daily ticket prices and piste lengths by country, examines the relationship between piste length and price, and plots the geocoded resort locations. Snowmaking variables are reported with their missing-value coverage because not every resort has a published snow-reliability report.

In [ ]:
eda_summary = df.groupby("country").agg(
    resorts=("resort_name", "count"),
    mean_price_eur=("price_day_eur", "mean"),
    median_price_eur=("price_day_eur", "median"),
    mean_slope_km=("slope_km", "mean"),
    median_slope_km=("slope_km", "median"),
    mean_lifts=("lifts_count", "mean"),
    median_rating=("rating_stars", "median"),
    median_snowmaking_pct=("snowmaking_pct", "median"),
)
print(eda_summary.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column="price_day_eur", by="country", ax=ax)
ax.set_title("Daily ski-pass price by country (EUR)")
ax.set_xlabel("Country")
ax.set_ylabel("Daily price (EUR)")
fig.suptitle("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column="slope_km", by="country", ax=ax)
ax.set_title("Ski slope length by country")
ax.set_xlabel("Country")
ax.set_ylabel("Total slope length (km)")
fig.suptitle("")
plt.tight_layout()
plt.show()

plot_data = df.dropna(subset=["slope_km", "price_day_eur"])
fig, ax = plt.subplots(figsize=(9, 5))
for country, group in plot_data.groupby("country"):
    ax.scatter(group["slope_km"], group["price_day_eur"], label=country, alpha=0.75)
ax.set_title("Resort size and daily ticket price")
ax.set_xlabel("Total slope length (km)")
ax.set_ylabel("Daily price (EUR)")
ax.legend(title="Country")
plt.tight_layout()
plt.show()

map_data = df.dropna(subset=["latitude", "longitude"])
fig, ax = plt.subplots(figsize=(9, 7))
for country, group in map_data.groupby("country"):
    ax.scatter(group["longitude"], group["latitude"], label=country, alpha=0.8)
ax.set_title("Geocoded Alpine ski resorts")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(title="Country")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


## 5. Export, coverage and final validation

The feature table retains all 280 Alpine resorts even when an external provider cannot resolve a coordinate or answer a hotel query. The coverage report distinguishes missing source information from failed external requests. Hotel counts from Overpass represent tagged map objects and may contain duplicate representations of one business.

In [ ]:
df.to_csv(FEATURES_CSV, index=False)

coverage_fields = [
    ("resort coordinates", df[["latitude", "longitude"]].notna().all(axis=1)),
    ("nearest-city distance", df["dist_nearest_city_km"].notna()),
    ("4/5-star hotel counts", df["hotels_4_5_star_10km"].notna()),
    ("EUR ticket price", df["price_day_eur"].notna()),
    ("overall Skiresort rating", df["rating_stars"].notna()),
    ("snow cannon count", df["snow_cannons"].notna()),
    ("snowmaking slope share", df["snowmaking_pct"].notna()),
    ("access village", df["nearest_access_village"].notna()),
    ("slope length", df["slope_km"].notna()),
    ("lift count", df["lifts_count"].notna()),
]
coverage = pd.DataFrame([
    {"feature": name, "available_rows": int(mask.sum()), "total_rows": len(df),
     "coverage_pct": round(float(mask.mean() * 100), 1)}
    for name, mask in coverage_fields
])
coverage.to_csv(COVERAGE_CSV, index=False)

print("Saved:", FEATURES_CSV)
print("Rows:", len(df))
print("\nCountry distribution:")
print(df.groupby("country").size())
print("\nFeature coverage:")
display(coverage)
print("\nHotel query status:")
print(df["hotel_query_status"].value_counts(dropna=False))
print("\nUnresolved coordinates:")
display(df.loc[df["latitude"].isna(), ["resort_name", "country", "nearest_access_village"]])
print("\nHotel batches:")
display(batch_summary)
assert len(df) == 280
assert df.groupby("country").size().eq(70).all()
assert df["alpine_scope_verified"].all()
assert df["url"].is_unique
assert os.path.exists(FEATURES_CSV)
